# Webinar 2: Data Preprocessing — Track 1: Tabular Pipeline
This notebook covers the complete tabular preprocessing lifecycle:
1. **Profiling**: Missing value audit, outlier diagnostics (IQR & Z-score), skewness, and cardinality.
2. **Encoding**: Handling dirty data, One-Hot Encoding for nominal columns, Ordinal Encoding for ordered categories.
3. **Scaling**: Comparing `StandardScaler`, `MinMaxScaler`, and `RobustScaler` on skewed features with extreme outliers.
4. **Imbalance**: Handling skewed class distribution using **SMOTE** (Synthetic Minority Over-sampling Technique).
5. **Train Model & Evaluation**: Comparing Baseline (unscaled, imbalanced) vs Preprocessed (encoded, robust-scaled, balanced) model performance.

In [ ]:
import sys
import os
# Ensure src is on python path
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.tabular import (
    profile_dataframe,
    print_profiling_report,
    clean_raw_tabular_data,
    TabularEncoder,
    TabularScaler,
    compare_scalers,
    balance_dataset,
    run_tabular_pipeline
)

pd.set_option('display.max_columns', None)
print('Imports loaded successfully!')

## Step 1: Data Profiling & Health Audit
Before applying transformations, we inspect missingness, statistical distributions, skewness, and extreme outliers.

In [ ]:
df_raw = pd.read_csv('../data/tabular/customer_churn_raw.csv')
print(f'Raw Tabular Dataset Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
profile = profile_dataframe(df_raw, target_col='churn')
print_profiling_report(profile)

## Step 2: Data Cleaning & Categorical Encoding
- Clean unphysical ages and string-formatted values in `total_charges` (`$`, spaces).
- Apply `OneHotEncoder` for nominal columns (`payment_method`, `internet_service`, `tech_support`).
- Apply `OrdinalEncoder` for ordered columns (`contract_type`: Month-to-month < One year < Two year).

In [ ]:
df_cleaned = clean_raw_tabular_data(df_raw)
X_clean = df_cleaned.drop(columns=['churn', 'customer_id'])
y = df_cleaned['churn'].map({'Yes': 1, 'No': 0})

encoder = TabularEncoder(
    nominal_cols=['payment_method', 'internet_service', 'tech_support'],
    ordinal_cols=['contract_type'],
    ordinal_categories={'contract_type': ['Month-to-month', 'One year', 'Two year']}
)
X_encoded = encoder.fit_transform(X_clean)
print(f'Encoded Feature Matrix Shape: {X_encoded.shape}')
X_encoded.head()

## Step 3: Feature Scaling Comparison
Compare `StandardScaler`, `MinMaxScaler`, and `RobustScaler` on numerical features with extreme outliers.

In [ ]:
scaler_comparison = compare_scalers(X_encoded, num_cols=['monthly_charges', 'age', 'tenure_months'])
scaler_comparison

In [ ]:
from src.evaluation.visualizer import plot_tabular_scaling_and_outliers
plot_tabular_scaling_and_outliers(df_raw, num_col='monthly_charges', output_path='../reports/tabular_scaling_and_outliers.png')

# Apply RobustScaler
scaler = TabularScaler(method='robust')
X_scaled = scaler.fit_transform(X_encoded)
X_scaled.head()

## Step 4: Handling Class Imbalance (SMOTE)
Customer churn is naturally imbalanced. We use SMOTE to oversample minority churn cases in feature space.

In [ ]:
print('Original Class Distribution:')
print(y.value_counts(normalize=True) * 100)

X_balanced, y_balanced = balance_dataset(X_scaled, y, method='smote')
print('\nBalanced Class Distribution (After SMOTE):')
print(y_balanced.value_counts())

## Step 5: Model Training & Before vs After Evaluation
Compare baseline model against the full preprocessed pipeline.

In [ ]:
tabular_results = run_tabular_pipeline('../data/tabular/customer_churn_raw.csv')

from src.evaluation.comparison import generate_modality_comparison
comp_df = generate_modality_comparison(tabular_results['baseline_metrics'], tabular_results['preprocessed_metrics'], 'Tabular')
comp_df